# Imports


In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
from pathlib import Path
from bs4 import BeautifulSoup

# Data Collection


### Get AnAge Data


In [47]:
page = 1
anage_data_url = (
    f"https://genomics.senescence.info/species/query.php?show=1&sort=3&page={page}"
)

try:
    res = requests.get(anage_data_url)
    res.raise_for_status()
    html = res.text

    tables = pd.read_html(html)

except requests.exceptions.RequestException as e:
    print(e)

/tmp/ipykernel_21180/1582643010.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


In [48]:
data_df = tables[1]
data_df.drop(columns=["All", "Display entry"], inplace=True)
data_df.rename(
    columns={
        "Species or taxon": "Species",
        "Common name": "Name",
        # "Longevity": "longevity",
        # "HAGRID": "ID",
    },
    inplace=True,
)
data_df

,HAGRID,Species,Name,Longevity
0,3242,Orycteropus afer,Aardvark,29.8
1,2161,Proteles cristata,Aardwolf,20
2,3296,Chilabothrus exsul,Abaco Island boa,21.7
3,570,Ciconia abdimii,Abdim's stork,21.6
4,3159,Sciurus aberti,Abert's squirrel,Not yet established
5,1180,Melozone aberti,Abert's towhee,8.6
6,2260,Genetta abyssinica,Abyssinian genet,Not yet established
7,3091,Thallomys paedulcus,Acacia rat,Not yet established
8,1323,Empidonax virescens,Acadian flycatcher,12.1
9,1419,Melanerpes formicivorus,Acorn woodpecker,17.2


In [49]:
species_cols_to_keep = [
    # "Species",
    # "Common name",
    "Kingdom",
    "Phylum",
    "Class",
    "Order",
    "Family",
    "Genus",
    "Maximum longevity",
    "Female sexual maturity",
    "Male sexual maturity",
    "Gestation",
    "Weaning",
    "Litter size",
    "Litters per year",
    "Inter-litter interval",
    "Weight at birth",
    "Weight at weaning",
    "Adult weight",
    "Postnatal growth rate",
    "Maximum longevity residual",
    "Typical body temperature",
    "Basal metabolic rate",
    "Body mass",
    "Metabolic rate per body mass",
]

In [50]:
data_copy = data_df.copy()
data_copy[species_cols_to_keep] = None
# data_copy

In [51]:
data_copy

,HAGRID,Species,Name,Longevity,Kingdom,Phylum,Class,Order,Family,Genus,...,Inter-litter interval,Weight at birth,Weight at weaning,Adult weight,Postnatal growth rate,Maximum longevity residual,Typical body temperature,Basal metabolic rate,Body mass,Metabolic rate per body mass
0,3242,Orycteropus afer,Aardvark,29.8,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2161,Proteles cristata,Aardwolf,20,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3296,Chilabothrus exsul,Abaco Island boa,21.7,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,570,Ciconia abdimii,Abdim's stork,21.6,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,3159,Sciurus aberti,Abert's squirrel,Not yet established,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5,1180,Melozone aberti,Abert's towhee,8.6,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
6,2260,Genetta abyssinica,Abyssinian genet,Not yet established,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
7,3091,Thallomys paedulcus,Acacia rat,Not yet established,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
8,1323,Empidonax virescens,Acadian flycatcher,12.1,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
9,1419,Melanerpes formicivorus,Acorn woodpecker,17.2,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
data_copy = data_df.copy()
data_copy[species_cols_to_keep] = None

for idx, row in data_copy.iterrows():
    species = row["Species"].replace(" ", "_")
    species_url = (
        f"https://genomics.senescence.info/species/entry.php?species={species}"
    )

    try:
        species_res = requests.get(species_url)
        species_res.raise_for_status()

        species_html = species_res.text

        species_soup = BeautifulSoup(species_html, "html.parser")
        dl_tags = species_soup.find_all("dl")

        # species_data_series = pd.Series(np.nan, index=species_cols_to_keep)
        # species_data = []

        species_dict = {}

        for dl_tag in dl_tags:
            dts = [dt.text.strip() for dt in dl_tag.find_all("dt")]
            dds = [dt.text.strip() for dt in dl_tag.find_all("dd")]

            species_dict.update(zip(dts, dds))

            # species_data_series = pd.concat(
            #     [species_data_series, pd.Series(species_dict)]
            # )

            if "Taxonomy" in species_dict:
                lines = species_dict["Taxonomy"].replace("\xa0", "").split("\n")
                for line in lines:
                    key, value = line.split(":", 1)
                    species_dict[key] = value

        # species_data_series = species_data_series[species_cols_to_keep]

        # data_copy.loc[idx] = species_data_series

        data_copy.loc[idx, species_dict.keys()] = species_dict.values()

    except requests.exceptions.RequestException as e:
        print(e)

    # break
display(data_copy)

,HAGRID,Species,Name,Longevity,Kingdom,Phylum,Class,Order,Family,Genus,...,Entrez,Ageing Literature,Images,Internet,Synonyms,,Incubation,Clutch size,Clutches per year,Weight at hatching
0,3242,Orycteropus afer,Aardvark,29.8,Animalia,Chordata,Mammalia (Taxon entry),Tubulidentata,Orycteropodidae,Orycteropus,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,NaN,NaN,NaN,NaN
1,2161,Proteles cristata,Aardwolf,20,Animalia,Chordata,Mammalia (Taxon entry),Carnivora,Hyaenidae,Proteles,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,Proteles cristatus,NaN,NaN,NaN,NaN,NaN
2,3296,Chilabothrus exsul,Abaco Island boa,21.7,Animalia,Chordata,Reptilia (Taxon entry),Squamata,Boidae,Chilabothrus,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,Epicrates exsul,No information is available on life history. P...,NaN,NaN,NaN,NaN
3,570,Ciconia abdimii,Abdim's stork,21.6,Animalia,Chordata,Aves (Taxon entry),Ciconiiformes,Ciconiidae,Ciconia,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,,(oviparous),,
4,3159,Sciurus aberti,Abert's squirrel,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Rodentia,Sciuridae,Sciurus,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,NaN,NaN,NaN,NaN
5,1180,Melozone aberti,Abert's towhee,8.6,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Passerellidae,Melozone,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,"Pipilo aberti, Kieneria aberti",NaN,14 days,3 (oviparous),1.5,3.63 g
6,2260,Genetta abyssinica,Abyssinian genet,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Carnivora,Viverridae,Genetta,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,NaN,NaN,NaN,NaN
7,3091,Thallomys paedulcus,Acacia rat,Not yet established,Animalia,Chordata,Mammalia (Taxon entry),Rodentia,Muridae,Thallomys,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,"Mus moggi, Thamnomys ruddi, Thallomys steven...",NaN,NaN,NaN,NaN,NaN
8,1323,Empidonax virescens,Acadian flycatcher,12.1,Animalia,Chordata,Aves (Taxon entry),Passeriformes,Tyrannidae,Empidonax,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,13 days,3 (oviparous),1,1.9 g
9,1419,Melanerpes formicivorus,Acorn woodpecker,17.2,Animalia,Chordata,Aves (Taxon entry),Piciformes,Picidae,Melanerpes,...,Search all databases,Search Google Scholar or Search PubMed,Google Image search,Search Google,NaN,NaN,14 days,4 (oviparous),2,4.7 g


In [ ]:
# lines = species_dict["Taxonomy"].replace("\xa0", "").split("\n")
# for line in lines:
#     key, value = line.split(":", 1)
#     species_data_series[key] = value

KeyError: 'Taxonomy'